### Ejercicio: Datos climáticos estación

Crea un nuevo DataFrame con datos de 1 estaciones SENAMHI de diferentes regiones del Perú (https://www.senamhi.gob.pe/site/descarga-datos/).

Considerar lo siguiente a partir de "tutorial-para-la-descarga-de-datos.pdf".

Los datos de las columnas corresponden a:
- Columna A: Año
- Columna B: Mes
- Columna C: Día
- Columna D: Precipitación acumulada
- Columna E: Temperatura máxima
- Columna F: Temperatura mínima

En las celdas donde aparece -99.9 significa que no hay información disponible para esa variable

Luego:
1. Parsear el nombres de la estación Chosica y leer el archivo.
1. Calcula el promedio mensual de precipitación
1. Calcula el promedio anual de precipitación
2. Identifica el día más lluvioso
4. Exporta los resultados formateados en CSV

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Tu código aquí – Ejercicio: Estación

# Leer los datos de las estaciones y crear un DataFrame combinado
import os
lista_archivos = os.listdir('/content/drive/MyDrive/Senamhi')  # Lista los archivos en el directorio
lista_archivos

['qc00000543_ñaña.txt',
 'qc00151209_chosica.txt',
 'qc00155213_santa_eulalia.txt',
 'qc00151205_CANCHACALLA.txt',
 'tutorial-para-la-descarga-de-datos.pdf',
 'qc00155224_santiago_de_tuna.txt']

In [5]:
# Parsear el nom_est del archivo y seleccionar la estación Chosica

def extraer_estacion(archivo_nombre):
    nombre_base = os.path.splitext(os.path.basename(archivo_nombre))[0]
    partes = nombre_base.split('_', 1)
    cod_est = partes[0]
    nom_est = partes[1] if len(partes) > 1 else nombre_base
    return cod_est, nom_est

archivos_chosica = [
    f for f in lista_archivos
    if 'chosica' in extraer_estacion(f)[1].lower()
]

if not archivos_chosica:
    raise FileNotFoundError(
        "No se encontró una estación cuyo nom_est contenga 'Chosica'. "
        "Revisa los nombres de los archivos en la carpeta SENAMHI."
    )

archivo_sel = archivos_chosica[0]
cod_est, nom_est = extraer_estacion(archivo_sel)

print(f"Código: {cod_est}")
print(f"Nombre: {nom_est}")
print(f"Archivo: {archivo_sel}")

Código: qc00151209
Nombre: chosica
Archivo: qc00151209_chosica.txt


In [8]:
# Leer el archivo CSV de la estación seleccionada

df_estacion = pd.read_csv(
    os.path.join(CARPETA_DATOS, archivo_sel),
    header=None,
    sep=r'\s+',
    na_values=['-999', '-99.9', -999, -99.9],
    engine='python'
)

df_estacion = df_estacion.iloc[:, :6].copy()
df_estacion.columns = ['Y', 'M', 'D', 'Pp', 'Tx', 'Tn']

for variable in df_estacion.columns:
    df_estacion[variable] = pd.to_numeric(df_estacion[variable], errors='coerce')

df_estacion = df_estacion.dropna(subset=['Y', 'M', 'D']).copy()

df_estacion['Fecha'] = pd.to_datetime(
    dict(
        year=df_estacion['Y'].astype(int),
        month=df_estacion['M'].astype(int),
        day=df_estacion['D'].astype(int)
    ),
    errors='coerce'
)

df_estacion.loc[df_estacion['Pp'] < 0, 'Pp'] = pd.NA

print(df_estacion.head())
print(f"\nRegistros válidos: {len(df_estacion)}")

      Y   M  D   Pp  Tx  Tn      Fecha
0  1989  12  1  0.0 NaN NaN 1989-12-01
1  1989  12  2  0.0 NaN NaN 1989-12-02
2  1989  12  3  0.0 NaN NaN 1989-12-03
3  1989  12  4  0.0 NaN NaN 1989-12-04
4  1989  12  5  0.0 NaN NaN 1989-12-05

Registros válidos: 8828


In [9]:
# Análisis de precipitación

# 1. Promedio mensual de precipitación
prom_lluvia_mes = (
    df_estacion.groupby('M', as_index=False)['Pp']
    .mean()
    .rename(columns={'Pp': 'Lluvia_promedio_mm'})
)

# 2. Precipitación total por año y promedio de los totales anuales
total_lluvia_anio = (
    df_estacion.groupby('Y', as_index=False)['Pp']
    .sum(min_count=1)
    .rename(columns={'Pp': 'Lluvia_total_mm'})
)

media_lluvia_anual = total_lluvia_anio['Lluvia_total_mm'].mean()

# 3. Día más lluvioso
pos_max = df_estacion['Pp'].idxmax()
dia_max_lluvia = None if pd.isna(pos_max) else df_estacion.loc[pos_max]

print(f"Precipitación mensual promedio:\n{prom_lluvia_mes}\n")
print(f"Precipitación anual:\n{total_lluvia_anio}\n")
print(f"Promedio anual de precipitación: {media_lluvia_anual:.2f} mm")

if dia_max_lluvia is not None:
    print(
        f"\nEl día más lluvioso fue el "
        f"{dia_max_lluvia['Fecha'].strftime('%Y-%m-%d')} "
        f"con {dia_max_lluvia['Pp']:.2f} mm de precipitación."
    )

# 4. Exportar resultados a CSV
prom_lluvia_mes.to_csv('/content/prom_lluvia_mes_chosica.csv', index=False)
total_lluvia_anio.to_csv('/content/total_lluvia_anio_chosica.csv', index=False)

if dia_max_lluvia is not None:
    pd.DataFrame([{
        'Fecha': dia_max_lluvia['Fecha'].strftime('%Y-%m-%d'),
        'Lluvia_mm': dia_max_lluvia['Pp']
    }]).to_csv('/content/dia_max_lluvia_chosica.csv', index=False)

print("\nArchivos CSV exportados en /content/")

Precipitación mensual promedio:
     M  Lluvia_promedio_mm
0    1            0.178629
1    2            0.307066
2    3            0.154278
3    4            0.084370
4    5            0.006732
5    6            0.000158
6    7            0.000840
7    8            0.000561
8    9            0.003184
9   10            0.004342
10  11            0.012899
11  12            0.044600

Precipitación anual:
       Y  Lluvia_total_mm
0   1989              0.0
1   1990             14.8
2   1991             14.0
3   1992              7.6
4   1993             14.6
5   1994             44.0
6   1995             19.6
7   1996             32.8
8   1997              0.9
9   1998             18.3
10  1999             39.6
11  2000             33.0
12  2001             25.7
13  2002             35.0
14  2003             21.5
15  2004              5.5
16  2005              5.9
17  2006             29.3
18  2007             15.7
19  2008             20.9
20  2009             51.9
21  2010              1

### Ejercicio 2: Datos climáticos multiestación

Crea un nuevo DataFrame con datos de 5 estaciones SENAMHI de diferentes regiones del Perú.

Luego:
1. Parsear los nombres de las estaciones y leer los archivos.
2. Unir las bases por fecha: `Fecha, Est1, Est2, ...`.
3. Calcular el promedio mensual de precipitación por estación/región.
4. Identificar el mes más lluvioso en cada estación.
5. Exportar el resultado formateado a CSV.

**Nota:** el código selecciona automáticamente 5 archivos disponibles, priorizando Chosica. Si el docente exige 5 regiones concretas, reemplaza `selected_files` por esos 5 archivos.

In [10]:
# Seleccionar 5 estaciones disponibles

archivos_sel = archivos_chosica[:1]

for f in lista_archivos:
    if f not in archivos_sel:
        archivos_sel.append(f)
    if len(archivos_sel) == 5:
        break

if len(archivos_sel) < 5:
    raise ValueError("Se necesitan al menos 5 archivos de estaciones SENAMHI.")

print("Estaciones seleccionadas:")
for f in archivos_sel:
    c, n = extraer_estacion(f)
    print(f"- {c}: {n}")

Estaciones seleccionadas:
- qc00151209: chosica
- qc00000543: ñaña
- qc00155213: santa_eulalia
- qc00151205: CANCHACALLA
- tutorial-para-la-descarga-de-datos: tutorial-para-la-descarga-de-datos


In [11]:
# Función para leer y limpiar una estación

def leer_estacion(ruta):
    datos = pd.read_csv(
        ruta,
        header=None,
        sep=r'\s+',
        na_values=['-999', '-99.9', -999, -99.9],
        engine='python'
    )

    datos = datos.iloc[:, :6].copy()
    datos.columns = ['Y', 'M', 'D', 'Pp', 'Tx', 'Tn']

    for variable in datos.columns:
        datos[variable] = pd.to_numeric(datos[variable], errors='coerce')

    datos = datos.dropna(subset=['Y', 'M', 'D']).copy()

    datos['Fecha'] = pd.to_datetime(
        dict(
            year=datos['Y'].astype(int),
            month=datos['M'].astype(int),
            day=datos['D'].astype(int)
        ),
        errors='coerce'
    )

    datos.loc[datos['Pp'] < 0, 'Pp'] = pd.NA
    return datos[['Fecha', 'Pp']].dropna(subset=['Fecha'])

In [14]:
# Leer las 5 estaciones y unirlas por fecha

series_estaciones = {}

# Filter out non-data files (like PDFs) from archivos_sel
archivos_data_validos = [f for f in archivos_sel if not f.lower().endswith('.pdf')]

for f in archivos_data_validos:
    cod_est, nom_est = extraer_estacion(f)
    datos = leer_estacion(os.path.join(CARPETA_DATOS, f))

    etiqueta = re.sub(
        r'[^A-Za-z0-9_ÁÉÍÓÚáéíóúÑñ-]',
        '',
        nom_est.replace(' ', '_')
    )

    series_estaciones[etiqueta] = datos.set_index('Fecha')['Pp']

df_multizona = pd.concat(series_estaciones, axis=1).reset_index()

print(df_multizona.head())
print(f"\nDimensiones: {df_multizona.shape}")

       Fecha  chosica  nana  santa_eulalia  CANCHACALLA
0 1963-12-01      NaN   NaN            0.0          NaN
1 1963-12-02      NaN   NaN            0.0          NaN
2 1963-12-03      NaN   NaN            0.0          NaN
3 1963-12-04      NaN   NaN            0.0          NaN
4 1963-12-05      NaN   NaN            0.2          NaN

Dimensiones: (18325, 5)


In [15]:
# Promedio mensual de precipitación y mes más lluvioso

prom_mes_estaciones = (
    df_multizona.assign(Mes=df_multizona['Fecha'].dt.month)
    .groupby('Mes')[list(series_estaciones.keys())]
    .mean()
)

print("Promedio mensual de precipitación (mm):")
display(prom_mes_estaciones)

meses_max_lluvia = pd.DataFrame({
    'Estacion': prom_mes_estaciones.columns,
    'Mes_mas_lluvioso': prom_mes_estaciones.idxmax().values,
    'Lluvia_promedio_mm': prom_mes_estaciones.max().values
})

print("\nMes más lluvioso por estación:")
display(meses_max_lluvia)

Promedio mensual de precipitación (mm):


,chosica,nana,santa_eulalia,CANCHACALLA
Mes,,,,
1,0.178629,0.026185,0.319776,1.849940
2,0.307066,0.033954,0.543553,3.080680
3,0.154278,0.020926,0.525434,2.781514
4,0.084370,0.001027,0.022222,0.555641
5,0.006732,0.000701,0.006583,0.021712
6,0.000158,0.002687,0.000142,0.000000
7,0.000840,0.001309,0.000000,0.000000
8,0.000561,0.000726,0.001317,0.000000
9,0.003184,0.000621,0.010278,0.022667



Mes más lluvioso por estación:


,Estacion,Mes_mas_lluvioso,Lluvia_promedio_mm
0,chosica,2,0.307066
1,nana,2,0.033954
2,santa_eulalia,2,0.543553
3,CANCHACALLA,2,3.080680


In [ ]:
# Exportar resultados a CSV

df_multizona.to_csv('/content/datos_5_estaciones.csv', index=False)
prom_mes_estaciones.to_csv('/content/prom_mes_5_estaciones.csv')
meses_max_lluvia.to_csv('/content/mes_max_lluvia_estacion.csv', index=False)

print("Archivos exportados correctamente.")